# We're going to process:

- geladas "gs://geladas"

In [1]:
import pandas as pd
import numpy as np

In [3]:
!uv pip install openpyxl

Using Python 3.10.16 environment at: /home/gagan/esp_projects/esp-data/.venv
Resolved 2 packages in 120ms                                         
Prepared 2 packages in 35ms                                              
Installed 2 packages in 9ms                                 
 + et-xmlfile==2.0.0
 + openpyxl==3.1.5


In [4]:
geladas_fileinfo = pd.read_excel("/home/gagan/open_audio_datasets/geladas/SharedMaterials_20200713/Gustison&Bergman_geladadata.xlsx", sheet_name="file_info", header=0)

In [5]:
geladas_fileinfo.head(3)

,focal_sample,focal_date,focal_time,focal_male,wav_file,wav_time,wav_state
0,MLG0001,2014-01-29,08:43:27,imp,MLG0001_call01,08:52:20,forage
1,MLG0003,2014-01-29,09:19:45,mhe,MLG0003_call01,09:33:24,rest
2,MLG0004,2014-01-29,10:39:19,dev,MLG0004_call01,10:39:54,forage


In [10]:
geladas_fileinfo.shape

(1515, 7)

In [6]:
geladas_call_log = pd.read_excel("/home/gagan/open_audio_datasets/geladas/SharedMaterials_20200713/Gustison&Bergman_geladadata.xlsx", sheet_name="call_log", header=0)

In [7]:
geladas_call_log.head(3)

,wav_file,caller_sex,caller_age,caller_id,vocal_type,vocal_onset,vocal_offset
0,MLG0001_call01,male,adult,imp,vg,0.978,1.303
1,MLG0001_call01,male,adult,imp,vg,1.668,1.964
2,MLG0001_call01,male,adult,imp,vg,2.522,2.818


In [11]:
geladas_call_log.shape

(10575, 7)

# Merge the call and file info dfs

In [8]:
geladas = pd.merge(left=geladas_fileinfo, right=geladas_call_log, how="outer", on="wav_file")

In [9]:
geladas.shape

(10583, 13)

In [12]:
geladas.head(5)

,focal_sample,focal_date,focal_time,focal_male,wav_file,wav_time,wav_state,caller_sex,caller_age,caller_id,vocal_type,vocal_onset,vocal_offset
0,MLG0001,2014-01-29,08:43:27,imp,MLG0001_call01,08:52:20,forage,male,adult,imp,vg,0.978,1.303
1,MLG0001,2014-01-29,08:43:27,imp,MLG0001_call01,08:52:20,forage,male,adult,imp,vg,1.668,1.964
2,MLG0001,2014-01-29,08:43:27,imp,MLG0001_call01,08:52:20,forage,male,adult,imp,vg,2.522,2.818
3,MLG0001,2014-01-29,08:43:27,imp,MLG0001_call01,08:52:20,forage,male,adult,imp,vg,3.204,3.465
4,MLG0001,2014-01-29,08:43:27,imp,MLG0001_call01,08:52:20,forage,female,adult,chu,vs,3.413,4.449


In [17]:
call_type_map = {
    "vb": "baby grunt",
    "vc": "copulation call",
    "vd": "display call",
    "ve": "whee inhale",
    "vf": "fear bark (submissive)",
    "vg": "exhaled contact grunt",
    "vi": "inhaled contact grunt",
    "vme": "exhaled moan",
    "vmi": "inhaled moan",
    "vp": "pre-copulation grunt",
    "vq": "infant squeal",
    "vs": "scream (submissive)",
    "vt": "threat grunt",
    "vwe": "exhaled wobble",
    "vwi": "inhaled wobble",
    "vx": "unknown",
    "vy": "vocal yawn",
    "vya": "first inhaled part of a vy",
    "vyb": "second exhaled part of a vy",
}


# Apply the call type mapping so that a sort of caption is used for vocal type

In [18]:
geladas["vocal_type_description"] = geladas["vocal_type"].apply(lambda x: call_type_map[x] if not pd.isna(x) else pd.NA)

# Rename wav_file to file_name

In [19]:
geladas = geladas.rename({"wav_file": "file_name"}, axis=1)
geladas.columns

Index(['focal_sample', 'focal_date', 'focal_time', 'focal_male', 'file_name',
       'wav_time', 'wav_state', 'caller_sex', 'caller_age', 'caller_id',
       'vocal_type', 'vocal_onset', 'vocal_offset', 'vocal_type_description'],
      dtype='object')

In [23]:
len(geladas["file_name"].unique())

1515

# Add .wav extension to file_name

In [24]:
geladas["file_name"] = geladas["file_name"].apply(lambda x: x + ".wav")

# Create a local_path column

In [27]:
geladas["local_path"] = ["audio/" + s for s in geladas["file_name"]]

In [28]:
geladas.shape

(10583, 15)

In [29]:
geladas.head(10)

,focal_sample,focal_date,focal_time,focal_male,file_name,wav_time,wav_state,caller_sex,caller_age,caller_id,vocal_type,vocal_onset,vocal_offset,vocal_type_description,local_path
0,MLG0001,2014-01-29,08:43:27,imp,MLG0001_call01.wav,08:52:20,forage,male,adult,imp,vg,0.978,1.303,exhaled contact grunt,audio/MLG0001_call01.wav
1,MLG0001,2014-01-29,08:43:27,imp,MLG0001_call01.wav,08:52:20,forage,male,adult,imp,vg,1.668,1.964,exhaled contact grunt,audio/MLG0001_call01.wav
2,MLG0001,2014-01-29,08:43:27,imp,MLG0001_call01.wav,08:52:20,forage,male,adult,imp,vg,2.522,2.818,exhaled contact grunt,audio/MLG0001_call01.wav
3,MLG0001,2014-01-29,08:43:27,imp,MLG0001_call01.wav,08:52:20,forage,male,adult,imp,vg,3.204,3.465,exhaled contact grunt,audio/MLG0001_call01.wav
4,MLG0001,2014-01-29,08:43:27,imp,MLG0001_call01.wav,08:52:20,forage,female,adult,chu,vs,3.413,4.449,scream (submissive),audio/MLG0001_call01.wav
5,MLG0001,2014-01-29,08:43:27,imp,MLG0001_call01.wav,08:52:20,forage,male,adult,imp,vi,3.732,4.176,inhaled contact grunt,audio/MLG0001_call01.wav
6,MLG0001,2014-01-29,08:43:27,imp,MLG0001_call01.wav,08:52:20,forage,male,adult,imp,vg,4.487,4.780,exhaled contact grunt,audio/MLG0001_call01.wav
7,MLG0001,2014-01-29,08:43:27,imp,MLG0001_call01.wav,08:52:20,forage,male,adult,imp,vg,5.157,5.453,exhaled contact grunt,audio/MLG0001_call01.wav
8,MLG0001,2014-01-29,08:43:27,imp,MLG0001_call01.wav,08:52:20,forage,male,adult,imp,vi,5.561,5.706,inhaled contact grunt,audio/MLG0001_call01.wav
9,MLG0001,2014-01-29,08:43:27,imp,MLG0001_call01.wav,08:52:20,forage,female,adult,chu,vs,5.590,6.344,scream (submissive),audio/MLG0001_call01.wav


# Add taxonomy

In [31]:
L = len(geladas)
kingdom = ["Animalia"] * L
phylum = ["Chordata"] * L
class_ = ["Mammalia"] * L
order = ["Primates"] * L
family = ["Circopithecidae"] * L
genus = ["Theropithecus"] * L
species_scientific = ["Theropithecus gelada"] * L
species_common = ["Gelada baboon"] * L

In [32]:
tax = pd.DataFrame({"kingdom": kingdom, "phylum": phylum, "class": class_, "order": order,
                    "family": family, "genus": genus, "species_scientific": species_scientific,
                    "species_common": species_common})

In [33]:
geladas = pd.concat([geladas, tax], axis=1)
geladas.columns

Index(['focal_sample', 'focal_date', 'focal_time', 'focal_male', 'file_name',
       'wav_time', 'wav_state', 'caller_sex', 'caller_age', 'caller_id',
       'vocal_type', 'vocal_onset', 'vocal_offset', 'vocal_type_description',
       'local_path', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus',
       'species_scientific', 'species_common'],
      dtype='object')

In [43]:
taxonomic_name = ["Chordata Mammalia Primates Circopithecidae Theropithecus gelada"] * L
taxonomic_name[:5]

['Chordata Mammalia Primates Circopithecidae Theropithecus gelada',
 'Chordata Mammalia Primates Circopithecidae Theropithecus gelada',
 'Chordata Mammalia Primates Circopithecidae Theropithecus gelada',
 'Chordata Mammalia Primates Circopithecidae Theropithecus gelada',
 'Chordata Mammalia Primates Circopithecidae Theropithecus gelada']

In [44]:
geladas["taxonomic_name"] = taxonomic_name

# Export

In [45]:
geladas.to_csv("/home/gagan/open_audio_datasets/geladas/SharedMaterials_20200713/geladas_annotations.csv", index=False)